In [1]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [2]:
train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')

train_df['text_lemma'] = train_df['text_lemma'].astype(str)
test_df['text_lemma'] = test_df['text_lemma'].astype(str)

In [3]:
train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')

train_df['text_lemma'] = train_df['text_lemma'].astype(str)
test_df['text_lemma'] = test_df['text_lemma'].astype(str)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

full_dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

In [4]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
def tokenize(batch):
    return tokenizer(
        batch["text_lemma"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized = full_dataset.map(tokenize, batched=True)
tokenized = tokenized.remove_columns(["text_lemma"])
tokenized.set_format("torch")

Map:   0%|          | 0/462447 [00:00<?, ? examples/s]

Map:   0%|          | 0/51384 [00:00<?, ? examples/s]

In [6]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy.compute(
            predictions=preds,
            references=labels
        )["accuracy"],
        "f1": f1.compute(
            predictions=preds,
            references=labels
        )["f1"]
    }

In [7]:
training_args = TrainingArguments(
    output_dir="./toxic_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    report_to="none"
)

In [8]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.117745,0.127829,0.964561,0.907521
2,0.100869,0.102921,0.971450,0.926381
3,0.061700,0.107458,0.974564,0.934555


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=86709, training_loss=0.11481903734496306, metrics={'train_runtime': 8637.0001, 'train_samples_per_second': 160.628, 'train_steps_per_second': 10.039, 'total_flos': 9.125618866354944e+16, 'train_loss': 0.11481903734496306, 'epoch': 3.0})

In [10]:
trainer.save_model("../toxic_model")
tokenizer.save_pretrained("../toxic_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./toxic_model\\tokenizer_config.json', './toxic_model\\tokenizer.json')

In [ ]:
def predict_toxicity(text: str):
    model.eval()

    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)

    return {
        "non_toxic": probs[0][0].item(),
        "toxic": probs[0][1].item()
    }